# CNN Classic Networks

从 LeNet 到 ResNet：为什么网络越做越深？残差连接如何让"深"成为可能？本课用梯度范数实验直接验证残差的作用，并训练一个小 CNN。


## 0. 环境配置与导入


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import matplotlib

matplotlib.rcParams["font.sans-serif"] = ["PingFang SC", "Hiragino Sans GB", "Arial Unicode MS", "Microsoft YaHei", "sans-serif"]
matplotlib.rcParams["axes.unicode_minus"] = False

print("PyTorch version:", torch.__version__)
torch.manual_seed(42)


## 1. 经典网络演进


| 网络 | 年份 | 层数(卷积) | 关键创新 |
|------|------|-----------|----------|
| LeNet-5 | 1998 | 2 | 卷积+池化+全连接范式 |
| AlexNet | 2012 | 5 | ReLU、Dropout、GPU、数据增强 |
| VGG | 2014 | 16-19 | 小核堆叠（3×3×3 ≈ 7×7） |
| GoogLeNet | 2014 | 22 | Inception 多尺度 |
| ResNet | 2015 | 50-152 | 残差连接 → 超深网络 |
| DenseNet | 2017 | 121+ | 密集连接 |

趋势：**更深 + 更宽 + 更高效的连接方式**。


## 2. 残差连接：让"深"成为可能


普通块学习映射 $F(x)$；残差块学习**残差** $H(x) = x + F(x)$：

$$y = x + F(x, W)$$

- 梯度多一条"恒等捷径"：$\partial y/\partial x = 1 + \partial F/\partial x$，深层梯度不会消失
- 极端情况下 $F \to 0$，网络退化为恒等——**深度增加不会变差**


In [ ]:
def block_grad_norms(use_residual, depth=12, seed=0):
    torch.manual_seed(seed)
    layers = [nn.Linear(16, 16) for _ in range(depth)]
    x = torch.randn(1, 16)
    h = x
    for lin in layers:
        h2 = torch.tanh(lin(h))
        h = h + h2 if use_residual else h2
    loss = h.pow(2).mean()
    norms = []
    for lin in layers:
        g = torch.autograd.grad(loss, lin.weight, retain_graph=True)[0]
        norms.append(g.norm().item())
    return np.array(norms)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, use_r in zip(axes, [False, True]):
    norms = block_grad_norms(use_r)
    ax.plot(norms, 'o-')
    ax.set_yscale('log')
    ax.set_xlabel('层（从输出往输入）'); ax.set_ylabel('|∂L/∂W| 范数')
    ax.set_title('残差连接' if use_r else '普通堆叠')
    ax.grid(alpha=0.3)
plt.tight_layout()
print("→ 普通堆叠：梯度随层数指数衰减；残差连接：梯度保持同一量级")


## 3. 残差块的实现


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn1 = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1)
        self.bn2 = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU()
    def forward(self, x):
        h = self.relu(self.bn1(self.conv1(x)))
        h = self.bn2(self.conv2(h))
        return self.relu(x + h)          # 残差捷径

x = torch.randn(2, 8, 16, 16)
y = ResidualBlock(8)(x)
print("输入:", tuple(x.shape), "→ 输出:", tuple(y.shape), "（分辨率与通道不变 ✓）")


## 4. 训练一个小 CNN（合成条形图分类）


离线生成 8×8 二分类图像数据（横条 vs 竖条），用小 CNN 训练——验证"卷积能学出结构特征"。


In [ ]:
rng = np.random.default_rng(0)
N = 800
X = np.zeros((N, 1, 8, 8)); y = np.zeros(N, dtype=np.int64)
for i in range(N):
    if rng.random() < 0.5:
        row = rng.integers(0, 8); X[i, 0, row, :] = 1.0; y[i] = 0
    else:
        col = rng.integers(0, 8); X[i, 0, :, col] = 1.0; y[i] = 1

fig, axes = plt.subplots(1, 4, figsize=(8, 2.4))
for i, ax in enumerate(axes):
    ax.imshow(X[i, 0], cmap='gray')
    ax.set_title('横条' if y[i] == 0 else '竖条')
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()


In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 4, 3, padding=1)
        self.pool1 = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(4, 8, 3, padding=1)
        self.pool2 = nn.MaxPool2d(2)
        self.fc = nn.Linear(8*2*2, 2)
    def forward(self, x):
        x = torch.relu(self.conv1(x))
        x = self.pool1(x)
        x = torch.relu(self.conv2(x))
        x = self.pool2(x)
        return self.fc(x.flatten(1))

torch.manual_seed(0)
model = TinyCNN()
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
Xt = torch.tensor(X, dtype=torch.float32); yt = torch.tensor(y)

losses, accs = [], []
for step in range(400):
    opt.zero_grad()
    idx = torch.randint(0, N, (64,))
    loss = F.cross_entropy(model(Xt[idx]), yt[idx])
    loss.backward(); opt.step()
    if step % 50 == 0:
        with torch.no_grad():
            acc = (model(Xt).argmax(1) == yt).float().mean().item()
        losses.append(loss.item()); accs.append(acc)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(losses, 'o-'); axes[0].set_xlabel('step/50'); axes[0].set_ylabel('loss'); axes[0].grid(alpha=0.3)
axes[1].plot(accs, 'o-'); axes[1].set_xlabel('step/50'); axes[1].set_ylabel('准确率'); axes[1].grid(alpha=0.3)
axes[0].set_title('训练损失'); axes[1].set_title('训练准确率')
print(f"最终准确率 = {accs[-1]:.3f}")


## 5. 从 8×8 到真实图像


真实任务（ImageNet 224×224、CIFAR 32×32）只是把这里的 8×8 换成真实数据集（`torchvision.datasets`），结构不变：

$$\text{conv}^{\times N} \to \text{pool} \to \text{conv}^{\times M} \to \text{pool} \to \text{flatten} \to \text{FC} \to \text{softmax}$$

现代实践用 `torchvision.models.resnet18(pretrained=True)` 直接迁移学习，但**手搭一遍**才能理解每个组件的作用。


## 课后练习


1. **改数据**：把条形图改成"左上亮 vs 右下亮"（对角线模式），CNN 还能学吗？
2. **加深**：堆 4 个残差块训练，对比收敛速度与准确率。
3. **去掉 BN**：把 BatchNorm 去掉重训，观察收敛变化（衔接 05 课初始化）。
4. **VGG 对比**：用 3 个 3×3 卷积（无残差）与 1 个 7×7 卷积比较参数数量。
5. **思考**：残差连接为什么对"梯度消失"的缓解作用那么强？（提示：恒等捷径的导数）
